# Turbopuffer RAG Training Pipeline

Fine-tune an LLM on your documents using reinforcement learning with Turbopuffer as the retrieval backend. This notebook chunks your documents, generates synthetic QA pairs, and launches a training job -- edit the configuration cell below and run top-to-bottom.

In [ ]:
# Uncomment for Google Colab
# !pip install cgft[turbopuffer] openai

In [ ]:
%load_ext autoreload
%autoreload 2

from cgft.utils import ensure_safe_python_version

ensure_safe_python_version()

## Configuration

In [ ]:
# Turbopuffer credentials -- https://turbopuffer.com/
TURBOPUFFER_API_KEY = ""
NAMESPACE = "my-namespace"
REGION = "aws-us-east-1"

# CGFT platform credentials -- https://app.cgft.io/account/api-keys
API_KEY = ""
BASE_URL = "https://app.cgft.io"

# LLM credentials -- used for QA generation, filtering, and refinement
LLM_API_KEY = ""
LLM_BASE_URL = ""
LLM_MODEL = "grok-4-fast-non-reasoning"

# Judge LLM credentials -- used at reward time to evaluate correctness + citation
JUDGE_API_KEY = LLM_API_KEY
JUDGE_BASE_URL = LLM_BASE_URL
JUDGE_MODEL = "grok-4-fast-non-reasoning"

# Documents to train on (local directory of markdown/text files)
DOCS_PATH = "./samples/posthog/docs/"

# Corpus context -- helps the QA generator understand your documents
CORPUS_DESCRIPTION = "Posthog documentation including docs, company policy, etc."
EXAMPLE_QUERIES = [
    "how to feature flag",
    "setup reverse proxy to avoid ad blockers",
    "posthog install nextjs",
]

# QA generation
TOTAL_SAMPLES = 100
OUTPUT_DIR = "outputs/tpuf_rag"

# Training experiment
EXPERIMENT_NAME = "my-search-v1"
EXPERIMENT_PREFIX = "tpuf-search"

## Step 1: Chunk & Upload Documents

In [ ]:
from cgft.corpus.turbopuffer.source import TpufChunkSource

source = TpufChunkSource(
    api_key=TURBOPUFFER_API_KEY,
    namespace=NAMESPACE,
    region=REGION,
)
source.populate_from_folder(DOCS_PATH)

## Step 2: Generate Synthetic QA

In [ ]:
from cgft.qa_generation.cgft_models import (
    CgftPipelineConfig,
    CorpusConfig,
    CorpusContextConfig,
    FilteringConfig,
    GenerationConfig,
    GroundingLLMFilterConfig,
    LLMDirectGenerationConfig,
    OutputConfig,
    PlatformConfig,
    RetrievalLLMFilterConfig,
    TargetsConfig,
)
from cgft.qa_generation.cgft_pipeline import CgftPipeline

cfg = CgftPipelineConfig(
    platform=PlatformConfig(
        api_key=API_KEY,
        base_url=BASE_URL,
        llm_api_key=LLM_API_KEY,
        llm_base_url=LLM_BASE_URL,
    ),
    corpus=CorpusConfig(corpus_name="rag-corpus", min_chunk_chars=400),
    corpus_context=CorpusContextConfig(
        description=CORPUS_DESCRIPTION,
        example_queries=EXAMPLE_QUERIES,
    ),
    generation=GenerationConfig(
        llm_direct=LLMDirectGenerationConfig(model=LLM_MODEL),
    ),
    filtering=FilteringConfig(
        retrieval_llm=RetrievalLLMFilterConfig(judge_model=LLM_MODEL),
        grounding_llm=GroundingLLMFilterConfig(judge_model=LLM_MODEL),
    ),
    targets=TargetsConfig(total_samples=TOTAL_SAMPLES),
    output=OutputConfig(dir=OUTPUT_DIR),
)
cfg.resolve_api_keys()

In [ ]:
pipeline = CgftPipeline(cfg, source_factory=lambda _cfg: source)
result = pipeline.run()

train_data = result["train_dataset"]
eval_data = result["eval_dataset"]

## Step 3: Launch Training

### Validate Environment

Run a local pre-flight check before submitting the training job. This simulates a full rollout (tool calls, reward computation) to catch bugs early.


In [ ]:
from cgft.trainer.validation import validate_env

validate_env(
    env_class=SearchEnv,
    env_args={
        "search": search,
        "judge_base_url": JUDGE_BASE_URL,
        "judge_api_key": JUDGE_API_KEY,
        "judge_model": JUDGE_MODEL,
    },
    train_dataset=train_data,
    eval_dataset=eval_data,
)


In [ ]:
import cgft
from cgft.corpus.turbopuffer.search import TpufSearch
from cgft.envs.search_env import SearchEnv
from cgft.trainer.pipeline import train

search = TpufSearch(
    api_key=TURBOPUFFER_API_KEY,
    namespace=NAMESPACE,
    region=REGION,
)

experiment_id = train(
    env_class=SearchEnv,
    env_args={
        "search": search,
        "judge_base_url": JUDGE_BASE_URL,
        "judge_api_key": JUDGE_API_KEY,
        "judge_model": JUDGE_MODEL,
    },
    train_dataset=train_data,
    eval_dataset=eval_data,
    prefix=EXPERIMENT_PREFIX,
    api_key=API_KEY,
    base_url=BASE_URL,
    local_modules=[cgft],
    pip_dependencies=["turbopuffer", "openai"],
    experiment_name=EXPERIMENT_NAME,
    validate_env_remotely=True,
)

## What to Expect

Training runs take 30-60 minutes. Monitor at [app.cgft.io](https://app.cgft.io).

**Early training**: Rewards will fluctuate and metrics may be noisy. This is normal -- the model is learning basic patterns.

**Healthy trajectory**: pass@k increases first (model solves more prompts), then average reward follows (model gets better at each prompt).

**Warning signs**:
- pass@k flat while reward increases -- overfitting to easy prompts
- Both stagnate early -- training distribution too hard or reward signal too sparse